# Fresh-start ingest

One-time clean rebuild of the corpus:
1. Wipe `chroma_db/` and `data/processed/*.json` (keep eval data + source PDFs)
2. Re-parse PDFs through Docling
3. Re-embed all chunks into Chroma with stable content-hash IDs

After this completes, the corpus is fully reproducible — same content always
produces the same chunk_ids, so re-parsing is idempotent.

⚠️ **Cell 2 won't actually delete anything until you set `CONFIRM_DELETE = True`.**

In [3]:
"""Wipe chroma_db/ and data/processed/*.json. Eval data is preserved.

Default mode = DRY RUN: shows what will be deleted without doing it.
Set CONFIRM_DELETE = True, re-run the cell to actually delete.
"""
import shutil
from rag_pipeline.config import cfg, log

CONFIRM_DELETE = False   # ← flip to True, then re-run, to actually delete

to_delete = [
    cfg.CHROMA_PERSIST_DIR,
    cfg.DATA_PROCESSED_DIR / "phase0_chunks.json",
    cfg.DATA_PROCESSED_DIR / "phase1_chunks.json",
    cfg.DATA_PROCESSED_DIR / "ragas_rows_mq.json",
    cfg.DATA_PROCESSED_DIR / "ragas_quickstart_cache.json",
]
to_keep = [
    cfg.EVAL_SET_PATH,
    cfg.PROJECT_ROOT / "src" / "rag_pipeline" / "eval" / "data",
    cfg.DATA_RAW_DIR,
    cfg.EVAL_RESULTS_DIR,
]

print("=== WILL DELETE ===")
for p in to_delete:
    if p.exists():
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"  {kind}  {p}")
    else:
        print(f"  (already gone)  {p}")

print("\n=== WILL KEEP ===")
for p in to_keep:
    mark = "✓" if p.exists() else "?"
    print(f"  {mark}  {p}")

if not CONFIRM_DELETE:
    print("\n⚠️  CONFIRM_DELETE=False — nothing deleted. Set True + re-run to wipe.")
else:
    print("\n🔥 Deleting...")
    for p in to_delete:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
            log.info(f"  rm -rf {p}")
        elif p.is_file():
            p.unlink()
            log.info(f"  rm {p}")
    print("\n✅ Clean slate ready")

=== WILL DELETE ===
  DIR   /home/thimu/github_vs/protoRAG/rag-pipeline/chroma_db
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase0_chunks.json
  FILE  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase1_chunks.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_rows_mq.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_quickstart_cache.json

=== WILL KEEP ===
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/eval_set.json
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/src/rag_pipeline/eval/data
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/data/raw
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/results

⚠️  CONFIRM_DELETE=False — nothing deleted. Set True + re-run to wipe.


In [4]:
"""

Intial Config, LLM, Embedding Model, Data Corpus, Paths ::::::::::::::::::::::::


Verify config + define shared constants for the rest of the notebook."""

from pathlib import Path
from rag_pipeline.config import cfg, log

log.info(f"Provider:    {cfg.MODEL_PROVIDER}")
log.info(f"LLM:         {cfg.OLLAMA_MODEL}")
log.info(f"Embeddings:  {cfg.OLLAMA_EMBEDDING_MODEL}")

# Where your IPC PDFs live (NOT under data/raw/ — they're in your Downloads folder)
PDF_SOURCE_DIR = Path("/home/thimu/Downloads/pdf_splitter/IPC")
assert PDF_SOURCE_DIR.exists(), f"PDFs not found at {PDF_SOURCE_DIR}"

CHUNKS_CACHE = cfg.DATA_PROCESSED_DIR / "phase1_chunks.json"
COLLECTION   = "IPC_Corpus"

log.info(f"PDF source:  {PDF_SOURCE_DIR}")
log.info(f"PDFs found:  {len(list(PDF_SOURCE_DIR.glob('*.pdf')))}")

2026-06-05 01:06:07,724 - INFO    | rag - Provider:    ollama
2026-06-05 01:06:07,725 - INFO    | rag - LLM:         gemma-4-e4b:latest
2026-06-05 01:06:07,725 - INFO    | rag - Embeddings:  embeddinggemma:latest
2026-06-05 01:06:07,725 - INFO    | rag - PDF source:  /home/thimu/Downloads/pdf_splitter/IPC
2026-06-05 01:06:07,726 - INFO    | rag - PDFs found:  74


In [6]:
"""

Parsing:::::::::::::::::::::

Parse every PDF under PDF_SOURCE_DIR with the production dispatcher.

Wall time: ~15-18 min for 63 IPC PDFs on CPU (Docling falls back from CUDA).
The resulting chunks are saved to phase1_chunks.json so this never has to
run again unless the source PDFs change.
"""
from rag_pipeline.parsers import default_dispatcher, save_chunks_cache

dispatcher = default_dispatcher()
chunks = dispatcher.parse_directory(PDF_SOURCE_DIR)

assert chunks, "Parsing produced 0 chunks — check PDF_SOURCE_DIR"

save_chunks_cache(chunks, CHUNKS_CACHE)
log.info(f"Parsed and cached {len(chunks)} chunks")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-06-05 01:06:29,337 - INFO    | rag - Found 74 supported files under /home/thimu/Downloads/pdf_splitter/IPC
Parsing: 100%|██████████| 74/74 [01:12<00:00,  1.02file/s]
2026-06-05 01:07:41,778 - INFO    | rag - Produced 610 unique chunks (after content-hash dedup)
2026-06-05 01:07:41,778 - INFO    | rag -   pdf: 610 chunks
2026-06-05 01:07:41,788 - INFO    | rag - Saved 610 chunks -> phase1_chunks.json
2026-06-05 01:07:41,788 - INFO    | rag - Parsed and cached 610 chunks


In [8]:
"""
Embedding:::::::::::::::::::


Create an empty Chroma collection and embed every chunk.

Because Cell 2 wiped chroma_db/, the collection starts empty — no need
for the idempotent dedup logic here. Every chunk gets embedded once.

"""

from rag_pipeline.vectorstore import get_vectorstore
from tqdm import tqdm

vs = get_vectorstore(COLLECTION)
assert vs._collection.count() == 0, "Collection not empty — did Cell 2 actually wipe?"

docs = [c.to_langchain_document() for c in chunks]
ids  = [c.chunk_id for c in chunks]

BATCH = 64
for i in tqdm(range(0, len(chunks), BATCH), desc="Embedding"):
    vs.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])

final_count = vs._collection.count()
log.info(f"Vectorstore '{COLLECTION}' has {final_count} vectors")
assert final_count == len(chunks), f"Expected {len(chunks)} vectors, got {final_count}"

AssertionError: Collection not empty — did Cell 2 actually wipe?

In [9]:
"""
Retriever:::::::::::::::::::::::::::::::::

hybrid_reranked = dense + BM25 + BGE cross-encoder rerank."""

from rag_pipeline.retrievers import (
    BM25Retriever, DenseRetriever, EnsembleRetriever,
    Reranker, RerankedRetriever,
)

dense    = DenseRetriever(collection_name=COLLECTION)
bm25     = BM25Retriever(chunks)
ensemble = EnsembleRetriever([dense, bm25], fetch_k=20)
reranker = Reranker()                                     # downloads BGE on first run
hybrid_r = RerankedRetriever(ensemble, reranker, fetch_k=20, min_score=0.5)

log.info("Retriever stack ready")

2026-06-05 01:12:49,450 - INFO    | rag - Loading reranker: BAAI/bge-reranker-base
2026-06-05 01:12:50,608 - INFO    | rag - Retriever stack ready


In [10]:
"""
Querying::::::::::::::::::::::::::::

Single-query smoke test through the full RAG pipeline."""

from rag_pipeline.providers import get_llm
from rag_pipeline.generation import answer, pretty_print

llm = get_llm()

response = answer(
    query="What is the punishment for cruelty by a husband against his wife?",
    retriever=hybrid_r,
    llm=llm,
    top_k=3,
)
pretty_print(response, score_label="rerank")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
2026-06-05 01:13:03,444 - INFO    | rag - PromptManager: loading from /home/thimu/github_vs/protoRAG/rag-pipeline/src/rag_pipeline/prompts/templates


What is the punishment for cruelty by a husband against his wife?

A person who is the husband or relative of the husband of a woman who subjects her to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine [2].

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_10.pdf  |  p.3 | § OF OFFENCES AGAINST WOMAN AND CHILD   (rerank=0.849)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_11.pdf  |  p.4 | § Of offences relating to marriage   (rerank=0.840)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf  |  p.3 | § OF OFFENCES AGAINST WOMAN AND CHILD Of sexual offences   (rerank=0.799)


In [11]:
"""

Retrieval Metrics::::::::::::::::::::::::::::

Score the fresh retriever against the eval set. ~30-60 sec."""

from rag_pipeline.eval import load_eval_set, evaluate_retriever

examples = load_eval_set(cfg.EVAL_SET_PATH)
results  = evaluate_retriever(hybrid_r, examples, top_k=5)

print(f"\n=== Retrieval metrics ({results['n_positive']} positives) ===\nOverall:")
for k, v in results["overall"].items():
    print(f"  {k:8s}: {v:.3f}")

if results["by_difficulty"]:
    print("\nBy difficulty:")
    for diff, m in results["by_difficulty"].items():
        row = "  ".join(f"{k}={v:.3f}" for k, v in m.items())
        print(f"  {diff:6s}: {row}")

2026-06-05 01:16:18,494 - INFO    | rag - Loaded 96 eval examples ← eval_set.json



=== Retrieval metrics (90 positives) ===
Overall:
  hit     : 0.411
  recall  : 0.383
  mrr     : 0.341
  snippet : 0.356

By difficulty:
  easy  : hit=0.950  recall=0.950  mrr=0.833  snippet=0.950
  medium: hit=0.433  recall=0.433  mrr=0.339  snippet=0.433
  hard  : hit=0.125  recall=0.062  mrr=0.096  snippet=0.000


In [12]:
"""
Refusal rate check:::::::::::::::::::::::::

Every negative should produce a refusal. Phase 4 baseline = 100%.

"""
from rag_pipeline.eval import load_negatives, is_refusal

negatives = load_negatives()
refusals = 0
for neg in negatives:
    resp = answer(neg.question, hybrid_r, llm, top_k=3)
    refused = is_refusal(resp["answer"])
    refusals += int(refused)
    print(f"  {'good' if refused else 'bad'}  {neg.question[:80]}")

print(f"\nRefusal rate: {refusals}/{len(negatives)} ({100 * refusals / len(negatives):.0f}%)")

2026-06-05 01:25:58,729 - INFO    | rag - Loaded 7 negative examples ← ipc_negatives.yaml
2026-06-05 01:26:01,040 - INFO    | rag - Loaded 6 refusal markers ← refusal_markers.yaml


  good  How many days of parental leave are employees entitled to in India?
  good  What is the current GST rate on luxury items?
  good  How many members must a startup team have?
  good  What is the procedure for filing an FIR under the CrPC?
  good  What does the IT Act say about cybercrime penalties?
  good  What does Article 21 of the Constitution guarantee?
  good  Under the Indian Evidence Act, what is the rule regarding hearsay?

Refusal rate: 7/7 (100%)


In [13]:
"""Compare the format/content of gold_source_paths in eval_set.json
vs source_path in the freshly-parsed chunks. If they differ, we found it."""

from rag_pipeline.config import cfg
from rag_pipeline.eval import load_eval_set
from rag_pipeline.parsers import load_chunks_cache

examples = load_eval_set(cfg.EVAL_SET_PATH)
chunks   = load_chunks_cache(cfg.PROJECT_ROOT / "data" / "processed" / "phase1_chunks.json")

# All unique source paths from the FRESH chunks
fresh_paths = sorted(set(c.source_path for c in chunks))
print(f"Fresh chunks: {len(chunks)} chunks across {len(fresh_paths)} files")
print(f"  Sample fresh paths:")
for p in fresh_paths[:3]:
    print(f"    {p!r}")

# All unique gold source paths referenced by the eval set
gold_paths = set()
for ex in examples:
    gold_paths.update(ex.gold_source_paths)
print(f"\nEval set references {len(gold_paths)} unique source paths")
print(f"  Sample gold paths:")
for p in sorted(gold_paths)[:3]:
    print(f"    {p!r}")

# How many gold paths actually exist in the fresh chunks?
matched = gold_paths & set(fresh_paths)
unmatched = gold_paths - set(fresh_paths)
print(f"\n=== MATCH SUMMARY ===")
print(f"  Gold paths matched in fresh chunks: {len(matched)} / {len(gold_paths)}")
print(f"  Gold paths NOT in fresh chunks:     {len(unmatched)}")
if unmatched:
    print(f"\n  First few unmatched gold paths:")
    for p in list(unmatched)[:5]:
        print(f"    {p!r}")

2026-06-05 01:37:47,641 - INFO    | rag - Loaded 96 eval examples ← eval_set.json
2026-06-05 01:37:47,645 - INFO    | rag - Loaded 610 chunks <- phase1_chunks.json


Fresh chunks: 610 chunks across 74 files
  Sample fresh paths:
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_0.pdf'
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_1.pdf'
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_10.pdf'

Eval set references 57 unique source paths
  Sample gold paths:
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_0.pdf'
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_1.pdf'
    '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_11.pdf'

=== MATCH SUMMARY ===
  Gold paths matched in fresh chunks: 57 / 57
  Gold paths NOT in fresh chunks:     0


In [14]:
"""hybrid_reranked = dense + BM25 + BGE cross-encoder rerank.

Two variants:
  - hybrid_r       : production retriever, min_score=0.5 (used for inference + refusal testing)
  - hybrid_r_eval  : same stack, no min_score filter (used ONLY for retrieval-quality metrics)
"""
from rag_pipeline.retrievers import (
    BM25Retriever, DenseRetriever, EnsembleRetriever,
    Reranker, RerankedRetriever,
)

dense    = DenseRetriever(collection_name=COLLECTION)
bm25     = BM25Retriever(chunks)
ensemble = EnsembleRetriever([dense, bm25], fetch_k=20)
reranker = Reranker()

# Production retriever — strict safety filter
hybrid_r       = RerankedRetriever(ensemble, reranker, fetch_k=20, min_score=0.5)

# Eval-only retriever — no filter, measures raw retrieval quality
hybrid_r_eval  = RerankedRetriever(ensemble, reranker, fetch_k=20, min_score=None)

log.info("✅ Retriever stacks ready: hybrid_r (prod) + hybrid_r_eval (raw retrieval)")

2026-06-05 01:37:51,645 - INFO    | rag - Loading reranker: BAAI/bge-reranker-base
2026-06-05 01:37:52,796 - INFO    | rag - ✅ Retriever stacks ready: hybrid_r (prod) + hybrid_r_eval (raw retrieval)


In [15]:
"""Score the eval-only retriever against the eval set (no safety filter)."""
from rag_pipeline.eval import load_eval_set, evaluate_retriever

examples = load_eval_set(cfg.EVAL_SET_PATH)
results  = evaluate_retriever(hybrid_r_eval, examples, top_k=5)   # ← use the eval variant

print(f"\n=== Retrieval metrics ({results['n_positive']} positives) ===\nOverall:")
for k, v in results["overall"].items():
    print(f"  {k:8s}: {v:.3f}")

if results["by_difficulty"]:
    print("\nBy difficulty:")
    for diff, m in results["by_difficulty"].items():
        row = "  ".join(f"{k}={v:.3f}" for k, v in m.items())
        print(f"  {diff:6s}: {row}")

2026-06-05 01:41:09,500 - INFO    | rag - Loaded 96 eval examples ← eval_set.json



=== Retrieval metrics (90 positives) ===
Overall:
  hit     : 0.956
  recall  : 0.783
  mrr     : 0.724
  snippet : 0.478

By difficulty:
  easy  : hit=1.000  recall=1.000  mrr=0.858  snippet=1.000
  medium: hit=0.933  recall=0.933  mrr=0.683  snippet=0.767
  hard  : hit=0.950  recall=0.562  mrr=0.687  snippet=0.000


In [17]:
"""Benchmark each pipeline stage on a 20-question sample.
~5 minutes for 20 queries × full RAG pipeline."""
import random
from rag_pipeline.eval import LatencyTracker

LatencyTracker.reset()

# Stratified sample so we measure across difficulty levels
rng = random.Random(42)
positives = [e for e in examples if not e.is_negative()]
by_diff = {"easy": [], "medium": [], "hard": []}
for e in positives:
    by_diff[e.difficulty].append(e)
benchmark_set = (
    rng.sample(by_diff["easy"],   7) +
    rng.sample(by_diff["medium"], 7) +
    rng.sample(by_diff["hard"],   6)
)

# Measure each pipeline stage separately
for ex in benchmark_set:
    # Stage 1: dense alone
    with LatencyTracker.timer("retrieve.dense"):
        dense.retrieve(ex.question, top_k=5)

    # Stage 2: BM25 alone
    with LatencyTracker.timer("retrieve.bm25"):
        bm25.retrieve(ex.question, top_k=5)

    # Stage 3: ensemble (dense + BM25 + RRF)
    with LatencyTracker.timer("retrieve.ensemble"):
        ensemble.retrieve(ex.question, top_k=5)

    # Stage 4: full hybrid_reranked (production retriever, no min_score)
    with LatencyTracker.timer("retrieve.hybrid_reranked"):
        hybrid_r_eval.retrieve(ex.question, top_k=5)

    # Stage 5: end-to-end answer (retrieval + LLM generation)
    with LatencyTracker.timer("answer.end_to_end"):
        answer(ex.question, hybrid_r, llm, top_k=5)

print(LatencyTracker.report())

tag                                 n      mean       p50       p95       p99       min       max
-------------------------------------------------------------------------------------------------
answer.end_to_end                  20 29430.4ms 28175.2ms 72440.3ms 78646.2ms  1404.9ms 80197.7ms
retrieve.bm25                      20     2.9ms     2.9ms     4.6ms     5.5ms     1.3ms     5.7ms
retrieve.dense                     20  1247.7ms  1514.5ms  1848.3ms  6436.9ms    66.6ms  7584.0ms
retrieve.ensemble                  20    74.2ms    73.2ms    84.0ms    84.5ms    66.7ms    84.6ms
retrieve.hybrid_reranked           20  1453.8ms  1441.9ms  1567.6ms  1579.5ms  1392.3ms  1582.5ms
